# A Simple Word Embedding Based Model

---
In this notebook, we'll continue working with the "PubMed 200k RCT dataset", but this time, we'll apply a model that's based on word *embeddings* rather than word *counts*.

Goals of the notebook are as follows:

- Load word embeddings and see how they can be used to convert text into a numeric array
- Train and evaluate a VSWEM model, in which all word embeddings in a given document are aggregated to create a fixed-length feature vector
- Train and evaluate a SWEM model, in which word embeddings are refined, then aggregated to create a fixed-length feature vector

We'll need Tensorflow to train SWEM, so you may want to complete this notebook in Colab.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import requests, shutil

The next couple of blocks will load the dataset. We've seen this code several times.

In [2]:
train_url = 'https://github.com/Franck-Dernoncourt/pubmed-rct/raw/master/PubMed_20k_RCT/train.txt?raw=true'
val_url = 'https://github.com/Franck-Dernoncourt/pubmed-rct/raw/master/PubMed_20k_RCT/dev.txt?raw=true'
test_url = 'https://github.com/Franck-Dernoncourt/pubmed-rct/raw/master/PubMed_20k_RCT/test.txt?raw=true'

In [3]:
def read_pubmed_rct(url):

    labels = []
    sentences = []

    with requests.get(url) as r:
        for line in r.iter_lines():
            fields = line.decode('utf-8').strip().split('\t')
            if len(fields) == 2:
                labels.append(fields[0])
                sentences.append(fields[1])

    return sentences, labels

s_train, l_train = read_pubmed_rct(train_url)
s_val, l_val = read_pubmed_rct(val_url)
s_test, l_test = read_pubmed_rct(test_url)

print('There are %i sentences in the training set' % len(s_train))
print('There are %i sentences in the validation set' % len(s_val))
print('There are %i sentences in the test set' % len(s_test))

There are 180040 sentences in the training set
There are 30212 sentences in the validation set
There are 30135 sentences in the test set


In [4]:
# Downsample, since the dataset is larger than we need it to be

s_train, l_train = s_train[:10000], l_train[:10000]
s_val, l_val = s_val[:3000], l_val[:3000]
s_test, l_test = s_test[:3000], l_test[:3000]

## Word Embeddings

Now let's load our word vectors/embeddings. We'll use 300-dimensional embeddings from the Stanford NLP group (i.e. GloVe). No need to worry about how we're loading the file, but it's important to note that we can look up the vector for `'word'` by writing `glove_dict['word']`.

In [5]:
response = requests.get(
    'https://github.com/mengelhard/mmci_applied_ds/raw/master/data/glove/ce3_glove.npy',
    stream=True)

with open('glove.npy', 'wb') as fin:
    shutil.copyfileobj(response.raw, fin)

glove_dict = np.load('glove.npy', allow_pickle=True).item()

## Prepare the sentences for modeling

The code below will convert each sentence in our dataset into an L by 300 array, where L is the number of words in the sentence for which we have embeddings.

In [6]:
label_dict = {'BACKGROUND': 0, 'OBJECTIVE': 1, 'METHODS': 2, 'RESULTS': 3, 'CONCLUSIONS': 4}

def embed_sentence(s):
    return np.array([glove_dict[w.lower()] for w in s.split() if w.lower() in glove_dict.keys()])


x_train, y_train = [embed_sentence(s) for s in s_train], [label_dict[l] for l in l_train]
x_val, y_val = [embed_sentence(s) for s in s_val], [label_dict[l] for l in l_val]
x_test, y_test = [embed_sentence(s) for s in s_test], [label_dict[l] for l in l_test]